# Session 10 · SQL with DuckDB

Work top to bottom and run each cell with **Shift+Enter**.

Cells under a **Your turn** heading contain `...` placeholders. Replace them, run the cell, then run the check cell beneath it. Checks print `PASS` or `NOT YET` and never stop the notebook.

The data is the **synthetic** 2025 season from Session 3 (`data/matches_2025_synthetic.csv`): invented results for practice only. Solutions are in the matching `_solutions` notebook.

DuckDB is an analytical database that runs **inside** Python: no server, no setup. It can query CSV and Parquet files directly, and even query pandas DataFrames by name. It is the fastest way to bring your SQL skills into a Python workflow.

**Before you start**, in the terminal at the bottom of VS Code: `uv add duckdb`.

In [31]:
def check(label, get, expected):
    """Print PASS or NOT YET. `get` is a lambda, so missing names or errors never stop the notebook."""
    try:
        got = get()
    except Exception as e:
        print(f"NOT YET  {label}: {type(e).__name__}: {e}")
        return
    try:
        ok = got.equals(expected) if hasattr(got, "equals") else bool(got == expected)
        if isinstance(expected, float) and isinstance(got, (int, float)):
            ok = abs(got - expected) < 1e-6
    except Exception:
        ok = False
    print(f"PASS     {label}" if ok else f"NOT YET  {label}: expected {expected!r}, got {got!r}")

In [32]:
import numpy as np
import pandas as pd

matches = pd.read_csv("../data/matches_2025_synthetic.csv", parse_dates=["date"])
matches = matches.assign(
    home_score=lambda d: d["home_goals"] * 6 + d["home_behinds"],
    away_score=lambda d: d["away_goals"] * 6 + d["away_behinds"],
    total_goals=lambda d: d["home_goals"] + d["away_goals"],
    margin=lambda d: d["home_score"] - d["away_score"],
)
matches.head(3)

,match_id,season,round,date,home_team,away_team,home_goals,home_behinds,away_goals,away_behinds,crowd,home_score,away_score,total_goals,margin
0,1,2025,1,2025-03-16,Collingwood,Carlton,12,12,19,11,14893.0,84,125,31,-41
1,2,2025,1,2025-03-16,Hawthorn,Brisbane,14,9,17,10,12927.0,93,112,31,-19
2,3,2025,1,2025-03-15,North Melbourne,Richmond,16,11,10,9,21940.0,107,69,26,38


In [33]:
import duckdb

def same_table(a, b):
    """True if two DataFrames hold the same values, ignoring int/float differences and the index."""
    try:
        pd.testing.assert_frame_equal(a.reset_index(drop=True), b.reset_index(drop=True),
                                      check_dtype=False, check_exact=False)
        return True
    except AssertionError as e:
        print(e)
        return False

teams_fixed = pd.read_csv("../data/teams.csv").replace({"team": {"Greater Western Sydney": "GWS"}})

## 1. Query a file directly

Put the file path in quotes where a table name would go. `.df()` returns the result as a pandas DataFrame.

In [34]:
duckdb.sql("""
    SELECT home_team, COUNT(*) AS home_games
    FROM '../data/matches_2025_synthetic.csv'
    GROUP BY home_team
    ORDER BY home_games DESC
    LIMIT 5
""").df()

,home_team,home_games
0,Essendon,16
1,Melbourne,15
2,Gold Coast,15
3,Richmond,14
4,GWS,13


### Your turn 1.1

Set `n_rows_sql` to the number of rows in the CSV file, using SQL. `.fetchone()` returns the first row as a tuple, so `.fetchone()[0]` gives a single value.

In [35]:
n_rows_sql = duckdb.sql("""
                    SELECT COUNT(*) AS NumRows 
                    FROM '../data/matches_2025_synthetic.csv'
""").limit(1).fetchall()[0][0]

In [36]:
check("1.1 n_rows_sql", lambda: n_rows_sql, 207)

PASS     1.1 n_rows_sql


## 2. Query a DataFrame by name

DuckDB can see any DataFrame in your session and query it as if it were a table. Here `matches` is the pandas DataFrame from the setup cell, including the score columns added there.

In [37]:
# Average margin by round
duckdb.sql("SELECT round, AVG(margin) AS avg_margin FROM matches GROUP BY round ORDER BY round LIMIT 5").df()

,round,avg_margin
0,1,-3.333333
1,2,16.111111
2,3,19.111111
3,4,4.222222
4,5,4.555556


### Your turn 2.1: the same answer two ways

Build a per-round summary with `round`, `games` (count of matches) and `avg_total_goals` (mean of `total_goals`), sorted by round:

- `sql_round`: in SQL with DuckDB
- `pd_round`: in pandas with `groupby`

The check confirms they hold the same values. Doing the same job twice and comparing is exactly the habit that makes Milestone 2 work.

In [38]:
# Using DuckDB to get the number of games and average total goals by round
sql_round = duckdb.sql("SELECT round, COUNT(*) AS games, AVG(total_goals) AS avg_total_goals FROM matches GROUP BY round ORDER BY round").df()
# Now using pandas
pd_round = matches.groupby("round", as_index=False).agg(games = ("match_id", "count"), avg_total_goals = ("total_goals", "mean")).sort_values("round")

In [39]:
check("2.1 SQL matches pandas", lambda: same_table(sql_round, pd_round), True)

PASS     2.1 SQL matches pandas


## 3. Window functions and QUALIFY

DuckDB supports `QUALIFY`, which filters on a window function without a subquery. This is the SQL version of `group_by() %>% slice_max()`.

### Your turn 3.1

Set `top_home` to the highest-scoring home team in each round: one row per round, with `round`, `home_team` and `home_score`, ordered by round. When two scores tie, take the lower `match_id`.

Hint: `QUALIFY ROW_NUMBER() OVER (PARTITION BY round ORDER BY home_score DESC, match_id) = 1`

In [40]:
# Using DuckDB to get the top home team by score for each round
top_home = duckdb.sql("""
    SELECT round, home_team, home_score
    FROM matches
    QUALIFY ROW_NUMBER() OVER (PARTITION BY round ORDER BY home_score DESC, match_id) = 1 
    ORDER BY round
""").df()

In [41]:
check("3.1 rows", lambda: len(top_home), 23)
check("3.1 round 1", lambda: top_home.loc[top_home["round"] == 1, "home_team"].iloc[0], 'North Melbourne')

PASS     3.1 rows
PASS     3.1 round 1


## 4. Joins in SQL

### Your turn 4.1

Repeat Session 9's interstate count in SQL: join `teams_fixed` to `matches` twice (aliases `h` and `a`), and count the matches where the home and away states differ. Set `n_interstate_sql` to the count. It should equal your pandas answer from Session 9.

In [42]:
n_interstate_sql = duckdb.sql("""
    SELECT COUNT(*)
        FROM matches m
        JOIN teams_fixed h ON m.home_team = h.team
        JOIN teams_fixed a ON m.away_team = a.team
        WHERE h.state <> a.state
""").limit(1).fetchall()[0][0]

In [43]:
check("4.1 n_interstate_sql", lambda: n_interstate_sql, 140)

PASS     4.1 n_interstate_sql


## 5. Parquet

Parquet is a compressed, typed, column-based file format: smaller than CSV, much faster to read, and it keeps dates as dates. Most data teams store analytical data this way. DuckDB writes and reads it with no extra packages.

In [44]:
import os

duckdb.sql("COPY (SELECT * FROM matches) TO 'matches.parquet' (FORMAT parquet)")

print("CSV bytes:    ", os.path.getsize("../data/matches_2025_synthetic.csv"))
print("Parquet bytes:", os.path.getsize("matches.parquet"))
duckdb.sql("DESCRIBE SELECT * FROM 'matches.parquet'").df().head(6)

CSV bytes:     12499
Parquet bytes: 9591


,column_name,column_type,null,key,default,extra
0,match_id,BIGINT,YES,None,None,None
1,season,BIGINT,YES,None,None,None
2,round,BIGINT,YES,None,None,None
3,date,TIMESTAMP,YES,None,None,None
4,home_team,VARCHAR,YES,None,None,None
5,away_team,VARCHAR,YES,None,None,None


Notice that `date` comes back as a date type, with no `parse_dates` needed, and that the computed score columns are stored too. Add `*.parquet` to your `.gitignore` if you would rather not commit generated files.

## Wrap-up

Run All, confirm every check passes, and commit from VS Code (message: `Session 10: SQL with DuckDB`).